# 06: Validating the whole BIDS dataset

**Who this is for:** Users of the Scully Center MEG lab.

This notebook runs the official [BIDS validator](https://bids-standard.github.io/bids-validator/) on your
**whole** BIDS dataset and lists what it finds, **errors first**. Run it once everything is done (Notebooks 01 through 05).

The validator only **reads** the dataset (it is shown the folder read-only), so this notebook changes
nothing in the data. The one file it writes is the Methods text of Section 5, in
`derivatives/bids-conversion/`. Re-run it whenever you like: after fixing something, or after adding a participant.

**What you will do:**

1. Load the packages and point the notebook at your dataset.
2. Look at what's in the dataset.
3. Run the validator and read its findings.
4. Find which notebook fixes each finding.
5. Get a Methods text for your paper, describing how the dataset was made.

> 🖥️ **Where to run this:** on the cluster (scotty). The validator runs from an Apptainer container, and
> containers don't run on a laptop.

### Running this notebook from the pipeline

**Run the two cells below first, also when you run the notebook yourself.** They define `from_config()`,
which the settings cells use. Run by hand, `config_file` stays `None` and they change nothing: the notebook
uses the settings you edit in its cells. They let the Pygers-MEG-to-BIDS pipeline (`run_bids-pipeline.sh`) run this notebook
with the settings from a study's YAML files instead.

In pipeline mode, each settings cell ends with a line like
`EVENT_ID = from_config("EVENT_ID")`, which replaces the value typed above it with the one from the
YAML files. Run by hand, that line leaves the value as you typed it.

In [ ]:
# Left as None when you run the notebook yourself. The pipeline replaces it with the path of its
# settings file (papermill adds a cell right below this one that sets it).
config_file = None

In [ ]:
# Nothing to edit here. In pipeline mode (config_file set above), this reads this notebook's settings
# from the pipeline's settings file. Run by hand, PIPELINE stays None and from_config() changes nothing.
from pathlib import Path   # file paths that work on macOS, Linux and Windows

NOTEBOOK_ID = "06"     # this notebook's section in the pipeline's settings file
PIPELINE = None            # None = running by hand
if config_file is not None:
    import yaml                                       # reads .yaml files
    with open(config_file, encoding="utf-8") as f:
        PIPELINE = yaml.safe_load(f)[NOTEBOOK_ID]     # a dict: setting name -> value
    print(f"Pipeline mode: notebook {NOTEBOOK_ID} settings from {config_file}")


def from_config(*names):
    """The named settings: from the pipeline's settings file in pipeline mode, unchanged otherwise.

    Example:  EVENT_ID, DROP_UNMAPPED = from_config("EVENT_ID", "DROP_UNMAPPED")

    In pipeline mode every name must be in the settings file: a missing one stops the notebook, so a
    setting can never silently keep this notebook's example value. Settings the file lists under
    "_paths" are turned into Paths.
    """
    if PIPELINE is None:
        values = [globals()[name] for name in names]          # run by hand: keep the values typed above
    else:
        missing = [name for name in names if name not in PIPELINE]
        if missing:
            raise KeyError(f"The pipeline settings file has no value for {missing} (notebook {NOTEBOOK_ID}).")
        paths = set(PIPELINE.get("_paths", []))
        values = [Path(PIPELINE[name]) if name in paths and PIPELINE[name] is not None else PIPELINE[name]
                  for name in names]
    return values[0] if len(values) == 1 else tuple(values)

## 0. Load the Python packages

In [ ]:
# "import x" makes package x usable; "import x as y" lets the code use the short name y;
# "from x import y" brings in only y.
import json                      # read the validator's report, which it prints as JSON
import re                        # text patterns (removes color codes from the validator's text output)
import shutil                    # finds Apptainer, the program that runs the validator container
import subprocess                # runs the validator container, as if typed in a terminal
from pathlib import Path         # Path("...") turns text into a file path that works on macOS, Linux and Windows

import pandas as pd              # tables: the validator's findings are shown as a pandas DataFrame
from IPython.display import display   # shows a table in the output, also in the middle of a cell
from mne_bids import print_dir_tree   # pretty-prints the folders of a BIDS dataset

# Make pandas show every column, and long file names in full.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 1. Settings ✏️ (edit this cell)

Edit the lines between the `EDIT THESE LINES` markers, then run the cell. It checks each path and stops
with a message if one is wrong.

In [ ]:
# ---- EDIT THESE LINES -------------------------------------------------------
# BIDS_ROOT: the top folder of your BIDS dataset (the same BIDS_ROOT as in notebooks 04 and 05).
BIDS_ROOT = Path("/jukebox/[yourpath]/auditory-oddball/bids")

# VALIDATOR_SIF: the BIDS validator container (pull it once: see the commands at the top of notebook 03).
VALIDATOR_SIF = Path("/jukebox/[yourpath]/singularity-containers/bids-validator/bids-validator-3.0.2.sif")

IGNORE_WARNINGS = False    # True: report errors only
SHOW_TEXT_REPORT = False   # True: also print the validator's own text report (long)
FAIL_ON_ERRORS = False     # True: stop with an error at the end if the validator found errors
SUMMARY_FILE = None        # a .json file to save the error/warning counts in (None = don't save)
# -----------------------------------------------------------------------------

# Pipeline mode: these come from the YAML files (run by hand: unchanged)
BIDS_ROOT, VALIDATOR_SIF, IGNORE_WARNINGS, SHOW_TEXT_REPORT, FAIL_ON_ERRORS, SUMMARY_FILE = from_config(
    "BIDS_ROOT", "VALIDATOR_SIF", "IGNORE_WARNINGS", "SHOW_TEXT_REPORT", "FAIL_ON_ERRORS", "SUMMARY_FILE")

# ---- Check the settings (nothing below here needs editing) ----------------------
# assert stops the notebook with the message if the condition is False.
assert "[yourpath]" not in str(BIDS_ROOT), "Set BIDS_ROOT above (it still contains [yourpath])."
assert BIDS_ROOT.is_dir(), f"BIDS_ROOT not found: {BIDS_ROOT}"
assert (BIDS_ROOT / "dataset_description.json").is_file(), (
    f"No dataset_description.json in {BIDS_ROOT}. Is this the TOP folder of the dataset? "
    "(Notebook 04 writes this file.)")
assert "[yourpath]" not in str(VALIDATOR_SIF), "Set VALIDATOR_SIF above (it still contains [yourpath])."
assert VALIDATOR_SIF.is_file(), (f"Validator container not found: {VALIDATOR_SIF} "
                                 "(see the pull commands at the top of notebook 03).")

# Find Apptainer. shutil.which() returns the full path to a program, or None if it isn't installed.
# Newer clusters call it "apptainer", older ones "singularity"; they work the same.
SING = shutil.which("apptainer") or shutil.which("singularity")
assert SING, ("Apptainer/Singularity not found. Run this notebook on the cluster (e.g. scotty); "
              "containers don't run on a laptop.")

print("Dataset   :", BIDS_ROOT)
print("Validator :", VALIDATOR_SIF)
print("Apptainer :", SING)

## 2. What's in the dataset?

Before validating, a quick look at what the dataset holds: the folder tree (top levels only), and a table
with one row per data folder. Check that every participant and session you expect is there, with an
`anat` folder (from notebooks 03/04) and a `meg` folder with eye-tracking files (from notebook 05).

In [ ]:
# The top levels of the tree: dataset files, then sub-*/ses-*/<datatype> folders.
print_dir_tree(BIDS_ROOT, max_depth=3)

# One row per data folder (anat, meg, ...) of each participant/session, with its number of files.
# rglob("*") lists everything below a folder; we keep the folders named like a BIDS datatype.
DATATYPES = ["anat", "meg", "func", "dwi", "fmap", "beh"]
rows = []
for folder in sorted(BIDS_ROOT.glob("sub-*/**/*")):
    if folder.is_dir() and folder.name in DATATYPES:
        parts = folder.relative_to(BIDS_ROOT).parts       # e.g. ("sub-01", "ses-01", "meg")
        files = [f for f in folder.iterdir() if f.is_file()]
        rows.append({
            "participant": parts[0],
            "session": parts[1] if len(parts) == 3 else "n/a",    # "n/a" if the dataset has no sessions
            "datatype": folder.name,
            "files": len(files),
            # eye-tracking files have "_physio" in their names (they live in meg/, next to the MEG data)
            "eye-tracking files": sum("_physio" in f.name for f in files) if folder.name == "meg" else "",
        })
print()
display(pd.DataFrame(rows))

## 3. Run the BIDS validator

Runs the official validator on `BIDS_ROOT` and lists its findings, **errors first**. Errors make the
dataset invalid and must be fixed (Section 4 says where); warnings are advice.

The validator runs from its container, `VALIDATOR_SIF`, with Apptainer (called Singularity on older
clusters). The dataset is made visible inside the container read-only, so the validator can read it but
never change it.

Warnings you should expect (none of them make the dataset invalid):

> **Split recordings:** a FIF file larger than 2 GB is written as several files (`_split-01_meg.fif`,
> `_split-02_meg.fif`, …). The validator checks each split file, so a per-file warning lists each split
> separately and the total number of warnings goes up. This is expected.

* `TOO_FEW_AUTHORS` (shown for `dataset_description.json`): only one author is listed in `CITATION.cff`
  (notebook 04, 11.4c). It stays if the dataset really has a single author.
* `JSON_KEY_RECOMMENDED` in `dataset_description.json`: `License` (the validator asks for it here even when
  `CITATION.cff` carries it), `HEDVersion` and `SourceDatasets` (not used in this dataset).
* `SIDECAR_KEY_RECOMMENDED` in `_meg.json`, for RECOMMENDED fields that are left out:
  * `DeviceSerialNumber` (notebook 04, 10.1b);
  * `CogAtlasID`, `CogPOID` (notebook 04, 11.4d);
  * on the empty-room itself: `Instructions` and `AssociatedEmptyRoom`;
  * fields that do not apply to OPM or are not recorded: `ContinuousHeadLocalization`, `HeadCoilFrequency`,
    `MaxMovement` (no head coils), `ECOGChannelCount`, `SEEGChannelCount`, `SubjectArtefactDescription`.
* `SIDECAR_KEY_RECOMMENDED` in the `anat/` sidecars (e.g. `SpoilingType`): optional MRI metadata the
  scanner doesn't record (the same warnings notebook 03 showed for its staging folder).

The validator also works outside the notebook, in a terminal on the cluster:

```bash
apptainer run --cleanenv -B <BIDS_ROOT>:/data:ro <VALIDATOR_SIF> /data
```

or in a browser at <https://bids-standard.github.io/bids-validator/> (the folder is read locally, nothing
is uploaded).

In [ ]:
def validator_cmd(*args):
    """The command that runs the validator container, followed by the validator's own arguments.

    run          start the container's own program (the BIDS validator)
    --cleanenv   keep this notebook's environment settings out of the container
    -B ...:ro    show BIDS_ROOT inside the container at /data, read-only ("ro"), so the validator can
                 read the dataset but never change it; the validator is then pointed at /data
    """
    return [SING, "run", "--cleanenv", "-B", f"{BIDS_ROOT.resolve()}:/data:ro", str(VALIDATOR_SIF), *args]


def strip_colors(text):
    """Remove the terminal color codes (such as \x1b[31m for red) from the validator's text."""
    return re.sub(r"\x1b\[[0-9;]*m", "", text)

# Ask the container for the validator's version, as if typed in a terminal, and keep what it prints
# (stdout). This leaves a record in the notebook of which validator checked the dataset.
version = strip_colors(subprocess.run(validator_cmd("--version"), capture_output=True, text=True).stdout).strip()
print(f"{version}  ({VALIDATOR_SIF.name})")

# --- Run it, asking for machine-readable JSON output ---------------------------
cmd = validator_cmd("/data", "--format", "json")   # /data = BIDS_ROOT inside the container
if IGNORE_WARNINGS:
    cmd.append("--ignoreWarnings")
result = subprocess.run(cmd, capture_output=True, text=True)
# Read the printed JSON into a dict. If it is not JSON (e.g. the validator failed), show what it said and stop.
try:
    report = json.loads(result.stdout)
except json.JSONDecodeError:
    print(strip_colors(result.stdout + result.stderr))
    raise RuntimeError(f"The validator did not return a JSON report (exit code {result.returncode}).")

# --- Summary of what the validator found in the dataset --------------------------
s = report["summary"]
print(f"Subjects: {s['subjects']}  Sessions: {s['sessions']}  Tasks: {s['tasks']}  "
      f"Files: {s['totalFiles']}  Schema: {s['schemaVersion']}")

# --- Issues: one row per issue, grouped by severity / code / field ---------------
issues = pd.DataFrame(report["issues"]["issues"])
n_errors = n_warnings = 0
if issues.empty:
    print("\n✓ No errors and no warnings.")
else:
    # Not every issue has a subCode; make missing ones "" so they can be grouped.
    issues["subCode"] = issues.get("subCode", pd.Series(dtype=str)).fillna("")
    # An ordered category, so that errors sort before warnings.
    issues["severity"] = pd.Categorical(issues["severity"], ["error", "warning", "ignore"], ordered=True)
    # One row per (severity, code, subCode): the number of files concerned, and up to 3 of their names.
    table = (issues.groupby(["severity", "code", "subCode"], observed=True)
                   .agg(n_files=("location", "nunique"),
                        files=("location", lambda x: ", ".join(sorted(set(x))[:3]) + (" …" if x.nunique() > 3 else "")))
                   .reset_index())
    n_errors = int((issues["severity"] == "error").sum())
    n_warnings = int((issues["severity"] == "warning").sum())
    print(f"\n{'✗' if n_errors else '✓'} {n_errors} error(s), {n_warnings} warning(s)")
    with pd.option_context("display.max_colwidth", 120):
        display(table)

    # For errors, the full list: every file, with the validator's explanation where it gives one.
    if n_errors:
        cols = [c for c in ["code", "subCode", "location", "issueMessage"] if c in issues.columns]
        print("\nEvery error, one row per file (Section 4 says where to fix each one):")
        with pd.option_context("display.max_colwidth", 200):
            display(issues.loc[issues["severity"] == "error", cols].drop_duplicates().reset_index(drop=True))

if SHOW_TEXT_REPORT:
    text = subprocess.run(validator_cmd("/data"), capture_output=True, text=True)
    print(strip_colors(text.stdout + text.stderr))

# --- Save the counts (the pipeline puts them on its summary page) and stop on errors if asked ---
if SUMMARY_FILE is not None:
    Path(SUMMARY_FILE).write_text(json.dumps({"validator": version, "errors": n_errors, "warnings": n_warnings}) + "\n")
    print(f"\nCounts saved to {SUMMARY_FILE}")
if FAIL_ON_ERRORS and n_errors:
    raise RuntimeError(f"The BIDS validator found {n_errors} error(s): see the table above.")

### 3.1 Placeholders left in the dataset

The validator can't tell that a value such as `EDIT ME` was never filled in: to it, that is just text. This
cell lists every `EDIT ME` that reached the dataset's files, typically from the dataset description (authors,
release date, license text), which can wait until you share the dataset.

To fix them: fill them in (pipeline: the dataset description at the end of `global-study.yaml`, and `text/`;
by hand: notebook 04, Section 11), run notebook 04 again with the dataset files written
(`write_dataset_files: true` / `WRITE_DATASET_FILES = True`), then this notebook.

In [ ]:
# Look through the dataset's text files (not derivatives/) for "EDIT ME", line by line.
# Only text formats are read: .json, .tsv, .txt, .md, .cff and README / CHANGES / LICENSE.
TEXT_SUFFIXES = {".json", ".tsv", ".txt", ".md", ".cff"}
leftovers = []
for f in sorted(BIDS_ROOT.rglob("*")):
    rel = f.relative_to(BIDS_ROOT)
    if not f.is_file() or rel.parts[0] == "derivatives":
        continue
    if f.suffix in TEXT_SUFFIXES or f.name in ("README", "CHANGES", "LICENSE"):
        for n, line in enumerate(f.read_text(encoding="utf-8", errors="ignore").splitlines(), 1):
            if "EDIT ME" in line:
                leftovers.append({"file": str(rel), "line": n, "text": line.strip()[:100]})

if leftovers:
    print(f"⚠️ {len(leftovers)} placeholder(s) to fill in before you share the dataset:")
    with pd.option_context("display.max_colwidth", 100):
        display(pd.DataFrame(leftovers))
else:
    print("✓ No EDIT ME placeholders left in the dataset.")

## 4. Where to fix what

The `location` of each finding is the file it is about. Fix it in the notebook that wrote that file, re-run
that notebook, then re-run this one:

| Files in the finding | Fix it in |
|---|---|
| `dataset_description.json`, `README`, `CHANGES`, `LICENSE`, `CITATION.cff`, `participants.tsv/.json`, `task-<label>_meg.json` | notebook 04, Section 11 |
| `*_meg.json`, `*_channels.tsv/.json`, `*_events.tsv`, `*_coordsystem.json` | notebook 04, Section 10 |
| `*_events.json` | notebook 04 (10.4); notebook 05 merges the eye-tracker's screen fields into it |
| `anat/` files | notebook 03 (the conversion), then notebook 04, Section 12 (the copy) |
| `*_scans.tsv` | notebook 04: the MEG rows (Sections 8–9) and the MRI rows (Section 12) |
| `beh/` files (`*_beh.tsv`, `*_beh.json`) | notebook 04, Section 13 (the PsychoPy logs and their column descriptions) |
| `*_physio.*`, `*_physioevents.*` | notebook 02 (the conversion) and notebook 05 (the sync, and the copy into `meg/`) |

## 5. A Methods text for your paper

Like fMRIPrep's "boilerplate", this writes a short **Methods** description of how the dataset was made:
the conversion of the MEG, eye-tracking, MRI and behavioral data (notebooks 01 to 05), the
anonymization and the validation (Section 3), with the software versions and the references to cite.

* It reads everything from the **dataset itself** (the sidecar files, the tables and the eye-sync
  reports in `derivatives/eyesync/`), so the numbers and versions are those of your data. The one
  exception is the eye2bids version, which is the one installed here.
* It describes **all participants at once**, assuming they were all converted the same way. A value that
  differs between recordings (e.g. the clock drift) is given as a median and range.
* Text in **[brackets]** is for you to complete or remove. Read the text before using it: it is a
  draft, not a finished section.
* Run Section 3 first, so the validator's result is included.
* The text is shown below and saved to `derivatives/bids-conversion/group_desc-methods.md` (replaced
  each time the cell runs). It is the only file this notebook writes.

In [ ]:
# Builds a short Methods text from what is in the dataset, like fMRIPrep's "boilerplate".
# It reads only the dataset's own files (sidecars, tables, the eye-sync reports in derivatives/),
# so the numbers and versions are those of the data, not of this computer. The text is shown here and
# saved to derivatives/bids-conversion/group_desc-methods.md (replaced each time this cell runs).
import statistics                     # median of a list of numbers
from datetime import date             # today's date, for the note at the end
from importlib import metadata        # the installed version of a Python package (for eye2bids)
from IPython.display import Markdown  # shows text with formatting (bold, lists) in the output


# ---- Small helpers --------------------------------------------------------------------------
def read_json(path):
    """A .json file as a dict (an empty dict if the file is missing)."""
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path.is_file() else {}


def describe(values, unit="", fmt="{:g}"):
    """Numbers from all recordings as one piece of text: "1200 Hz" when they are all the same,
    "1200 Hz (range 1000-1200 Hz)" with the median first when they differ. None if there are none.
    fmt sets how each number is written, e.g. "{:.1f}" = one decimal."""
    values = [v for v in values if v is not None]
    if not values:
        return None
    if all(v == values[0] for v in values):
        return f"{fmt.format(values[0])}{unit}"
    return (f"{fmt.format(statistics.median(values))}{unit} "
            f"(range {fmt.format(min(values))}–{fmt.format(max(values))}{unit})")


def words(items):
    """["a", "b", "c"] -> "a, b and c" (each item only once, in order of first appearance)."""
    items = list(dict.fromkeys(i for i in items if i))
    return items[0] if len(items) == 1 else ", ".join(items[:-1]) + " and " + items[-1] if items else ""


def plural(n, word):
    """plural(1, "participant") -> "1 participant"; plural(3, ...) -> "3 participants"."""
    return f"{n} {word}{'' if n == 1 else 's'}"


# ---- 1. What the dataset holds ----------------------------------------------------------------
dataset = read_json(BIDS_ROOT / "dataset_description.json")
subjects = sorted(p.name for p in BIDS_ROOT.glob("sub-*") if p.is_dir() and p.name != "sub-emptyroom")
# Every file below the participants' folders (sub-emptyroom left out)
files = [f for s in subjects for f in sorted((BIDS_ROOT / s).rglob("*")) if f.is_file()]
# The task recordings' sidecars (_meg.json), without the empty-room recordings (task-noise)
meg_jsons = [f for f in files if f.name.endswith("_meg.json") and "_task-noise_" not in f.name]
megs = [read_json(f) for f in meg_jsons]
tasks = sorted({m.get("TaskName") for m in megs if m.get("TaskName")})
refs = []          # the references cited, added as the text is written

# The versions of the conversion tools, read from the dataset itself
mne_bids_version = next((g.get("Version") for g in dataset.get("GeneratedBy", [])
                         if g.get("Name") == "MNE-BIDS"), None)
readme = (BIDS_ROOT / "README").read_text(encoding="utf-8") if (BIDS_ROOT / "README").is_file() else ""
match = re.search(r"\bMNE (\d[\w.]*)", readme)            # e.g. "MNE 1.13.2" in the README
mne_version = match.group(1) if match else None
cmeg2fif_versions = [m.group(1) for meg in megs
                     if (m := re.search(r"cMEG2FIF\s+v?(\d[\w.]*)", meg.get("SoftwareVersions", "")))]

paragraphs = []

# ---- 2. Data organization ------------------------------------------------------------------
paragraphs.append(
    f"**Data organization.** The data were organized according to the Brain Imaging Data Structure "
    f"(BIDS, version {dataset.get('BIDSVersion', '[version]')}; Gorgolewski et al., 2016), including its MEG "
    f"extension (Niso et al., 2018), with the Pygers-MEG-to-BIDS notebooks "
    f"(https://github.com/mrribbits/pygers-meg). The dataset contains {plural(len(subjects), 'participant')} "
    f"and {plural(len(megs), 'task recording')} ({words('task-' + t for t in tasks)}).")
refs += ["Gorgolewski2016", "Niso2018"]

# ---- 3. OPM-MEG --------------------------------------------------------------------------------
if megs:
    bads = []                                   # number of bad channels in each task recording
    for f in meg_jsons:
        channels = f.with_name(f.name.replace("_meg.json", "_channels.tsv"))
        if channels.is_file():
            bads.append(int((pd.read_csv(channels, sep="\t")["status"] == "bad").sum()))
    # The event names, from the first task recording's _events.json
    events_json = read_json(meg_jsons[0].with_name(meg_jsons[0].name.replace("_meg.json", "_events.json")))
    event_names = list(events_json.get("trial_type", {}).get("Levels", {}))
    # Anonymized? Then every recording time in the scans.tsv files is before 1925 (notebook 04, Section 7)
    years = [int(t[:4]) for f in files if f.name.endswith("_scans.tsv")
             for t in pd.read_csv(f, sep="\t", dtype=str)["acq_time"].dropna() if t[:4].isdigit()]
    text = (
        f"**OPM-MEG.** MEG data were recorded with a {words(m.get('ManufacturersModelName') for m in megs)} "
        f"({describe([m.get('MEGChannelCount') for m in megs])} MEG and "
        f"{describe([m.get('MEGREFChannelCount') for m in megs])} reference channels, sampled at "
        f"{describe([m.get('SamplingFrequency') for m in megs], ' Hz')}; power-line frequency "
        f"{describe([m.get('PowerLineFrequency') for m in megs], ' Hz')}). The raw recordings were converted "
        f"to the FIF format with cMEG2FIF {words(cmeg2fif_versions) or '[version]'} "
        f"(https://github.com/mrribbits/cMEG2FIF-Scully), which stores the sensor positions, the device-to-head "
        f"transformation from the coregistration fiducials and the digitized head shape; no filtering or other "
        f"processing was applied.")
    if all(m.get("AssociatedEmptyRoom") for m in megs):
        text += " Each task recording has an associated empty-room recording from the same session."
    text += (" Flat or unusually noisy channels were flagged automatically and confirmed by inspecting their "
             "time courses and power spectra; "
             + ("no channels were marked bad." if bads and not any(bads) else
                f"{plural(bads[0], 'channel')} per recording {'was' if bads[0] == 1 else 'were'} marked bad."
                if bads and len(set(bads)) == 1 else
                f"{describe(bads) or '[n]'} channels per recording were marked bad."))
    if event_names:
        text += f" Events ({words(event_names)}) were decoded from the trigger channels."
    text += (f" The data were written to BIDS with MNE-BIDS {mne_bids_version or '[version]'} (Appelhoff et al., "
             f"2019) and MNE-Python {mne_version or '[version]'} (Gramfort et al., 2013).")
    refs += ["Appelhoff2019", "Gramfort2013"]
    if years and max(years) < 1925:
        text += (" To anonymize the data, all recording dates were shifted back by the same number of days "
                 "(keeping the time between recordings), and the operator name and subject information "
                 "were removed from the file headers.")
    else:
        text += " [The recording dates were not shifted: the dataset is not anonymized.]"
    paragraphs.append(text)

# ---- 4. Eye tracking ---------------------------------------------------------------------------
physios = [read_json(f) for f in files if f.name.endswith("_physio.json")]
if physios:
    def model(p):
        """"EYELINK II CL v5.51 ... (EyeLink 1000 Plus)" -> "EyeLink 1000 Plus" (the part in parentheses)."""
        name = p.get("ManufacturersModelName", "")
        m = re.search(r"\(([^()]*)\)\s*$", name)
        return m.group(1) if m else (name or p.get("Manufacturer"))
    methods = {"P-CR": "pupil and corneal reflection"}
    try:   # the eye2bids installed here, with its git commit if installed from GitHub
        commit = json.loads(metadata.distribution("eye2bids").read_text("direct_url.json") or "{}") \
            .get("vcs_info", {}).get("commit_id", "")
        eye2bids_version = f"version {metadata.version('eye2bids')}" + (f", commit {commit[:7]}" if commit else "")
    except metadata.PackageNotFoundError:
        eye2bids_version = "[version]"
    text = (
        f"**Eye tracking.** Gaze position and pupil size of the {words(p.get('RecordedEye') for p in physios)} "
        f"eye were recorded with an {words(model(p) for p in physios)} at "
        f"{describe([p.get('SamplingFrequency') for p in physios], ' Hz')} "
        f"({words(methods.get(p.get('EyeTrackingMethod'), p.get('EyeTrackingMethod')) for p in physios)}; "
        f"{words(p.get('CalibrationType') for p in physios)} calibration, average calibration error "
        f"{describe([p.get('AverageCalibrationError') for p in physios], '°', '{:.2f}')}). The EyeLink recordings "
        f"were converted to BIDS with eye2bids (Scully Center fork, {eye2bids_version}; "
        f"https://github.com/mrribbits/eye2bids).")
    # The clock alignment (notebook 05): one sync report per recording in derivatives/eyesync
    syncs = [read_json(f) for f in sorted((BIDS_ROOT / "derivatives" / "eyesync").rglob("*_desc-sync_report.json"))]
    if syncs:
        text += (
            f" To align the eye tracker with the MEG, the messages sent to the eye tracker during the task "
            f"were matched with the triggers recorded by the MEG "
            f"({describe([s.get('NumberOfPairs') for s in syncs])} pairs per recording), and a straight line "
            f"(offset and clock drift) was fitted to their times. The offset gives the eye-tracking data's "
            f"StartTime; the clock drift was {describe([s.get('DriftPPM') for s in syncs], ' ppm', '{:.1f}')} and "
            f"the residuals had a standard deviation of "
            f"{describe([s.get('ResidualSDms') for s in syncs], ' ms', '{:.1f}')}. A copy of the MEG data with "
            f"the eye-tracking channels resampled to the MEG clock (mne.preprocessing.realign_raw) is in "
            f"derivatives/eyesync.")
    paragraphs.append(text)

# ---- 5. MRI ------------------------------------------------------------------------------------
anats = [(f, read_json(f)) for f in files if f.parent.name == "anat" and f.suffix == ".json"]
if anats:
    names = {"T1w": "T1-weighted", "T2w": "T2-weighted", "FLAIR": "FLAIR"}
    kinds = [names.get(f.stem.rsplit("_", 1)[-1], f.stem.rsplit("_", 1)[-1]) for f, _ in anats]
    a = [j for _, j in anats]
    scanner = words(f"{j.get('MagneticFieldStrength', '[field]')} T {j.get('Manufacturer', '')} "
                    f"{j.get('ManufacturersModelName', '')}".strip() for j in a)
    paragraphs.append(
        f"**MRI.** {words(kinds)} images ({describe([j.get('SliceThickness') for j in a], ' mm')} thick slices) "
        f"were acquired on a {scanner} scanner and converted from DICOM to NIfTI with HeuDiConv "
        f"{words(j.get('HeudiconvVersion') for j in a) or '[version]'} (Halchenko et al., 2024) and dcm2niix "
        f"{words(j.get('ConversionSoftwareVersion') for j in a) or '[version]'} (Li et al., 2016). The images "
        f"in the dataset are not defaced, because MEG source modeling needs the whole head. [If you share "
        f"defaced images instead: The anatomical images were defaced with pydeface.]")
    refs += ["Halchenko2024", "Li2016"]

# ---- 6. Behavior -------------------------------------------------------------------------------
beh = [f for f in files if f.name.endswith("_beh.tsv")]
if beh:
    tables = [m.group(1) for f in beh if (m := re.search(r"_acq-([A-Za-z0-9]+)_", f.name))]
    presentation = (events_json.get("StimulusPresentation", {}) if megs else {})
    software = " ".join(filter(None, [presentation.get("SoftwareName"), presentation.get("SoftwareVersion")]))
    paragraphs.append(
        f"**Behavior.** The trial-by-trial logs of the experiment"
        + (f" ({software}; Peirce et al., 2019)" if "PsychoPy" in software else "")
        + f" were included as behavioral tables ({words(tables)}).")
    if "PsychoPy" in software:
        refs.append("Peirce2019")

# ---- 7. Validation (from Section 3, if it ran) ----------------------------------------------
if "n_errors" in globals():
    paragraphs.append(f"**Validation.** The dataset was checked with the BIDS validator ({version}), which "
                      f"reported {plural(n_errors, 'error')}.")
else:
    paragraphs.append("**Validation.** [Run Section 3 first to add the BIDS validator's result.]")

# ---- 8. The references, and the whole text ------------------------------------------------
REFERENCES = {
    "Gorgolewski2016": "Gorgolewski, K. J., et al. (2016). The brain imaging data structure, a format for organizing and describing outputs of neuroimaging experiments. *Scientific Data*, 3, 160044. https://doi.org/10.1038/sdata.2016.44",
    "Niso2018": "Niso, G., et al. (2018). MEG-BIDS, the brain imaging data structure extended to magnetoencephalography. *Scientific Data*, 5, 180110. https://doi.org/10.1038/sdata.2018.110",
    "Appelhoff2019": "Appelhoff, S., et al. (2019). MNE-BIDS: Organizing electrophysiological data into the BIDS format and facilitating their analysis. *Journal of Open Source Software*, 4(44), 1896. https://doi.org/10.21105/joss.01896",
    "Gramfort2013": "Gramfort, A., et al. (2013). MEG and EEG data analysis with MNE-Python. *Frontiers in Neuroscience*, 7, 267. https://doi.org/10.3389/fnins.2013.00267",
    "Halchenko2024": "Halchenko, Y. O., et al. (2024). HeuDiConv — flexible DICOM conversion into structured directory layouts. *Journal of Open Source Software*, 9(99), 5839. https://doi.org/10.21105/joss.05839",
    "Li2016": "Li, X., Morgan, P. S., Ashburner, J., Smith, J., & Rorden, C. (2016). The first step for neuroimaging data analysis: DICOM to NIfTI conversion. *Journal of Neuroscience Methods*, 264, 47–56. https://doi.org/10.1016/j.jneumeth.2016.03.001",
    "Peirce2019": "Peirce, J., et al. (2019). PsychoPy2: Experiments in behavior made easy. *Behavior Research Methods*, 51, 195–203. https://doi.org/10.3758/s13428-018-01193-y",
}
METHODS = "\n\n".join(
    ["### Methods: BIDS conversion"]
    + paragraphs
    + ["*This text was generated by notebook 06 on " + date.today().isoformat() + " from the files in the "
       "dataset. It describes all participants at once, as they were all converted the same way; a value "
       "that differs between recordings is given as a median and range. Text in [brackets] is for you to "
       "complete or remove. Check it before use, then copy it into your paper as it is or edited.*",
       "**References**"]
    + ["\n".join("- " + REFERENCES[r] for r in dict.fromkeys(refs))])   # one list, without blank lines

# ---- 9. Save it in derivatives/bids-conversion/ --------------------------------------------
# A derivatives folder is a small dataset of its own, so it gets a dataset_description.json too.
METHODS_DIR = BIDS_ROOT / "derivatives" / "bids-conversion"
METHODS_DIR.mkdir(parents=True, exist_ok=True)              # make the folder if it isn't there yet
(METHODS_DIR / "dataset_description.json").write_text(json.dumps({
    "Name": "Methods text for the BIDS conversion",
    "BIDSVersion": dataset.get("BIDSVersion", "1.11.1"),
    "DatasetType": "derivative",
    "GeneratedBy": [{"Name": "06_Validate_BIDS.ipynb (Pygers-MEG-to-BIDS)",
                     "Description": "A draft Methods text describing how this BIDS dataset was made"}],
}, indent=4) + "\n", encoding="utf-8")
METHODS_FILE = METHODS_DIR / "group_desc-methods.md"
METHODS_FILE.write_text(METHODS + "\n", encoding="utf-8")

display(Markdown(METHODS))      # the text, formatted
print(f"\nSaved to {METHODS_FILE}")
print("\n----- The same text, to copy (Markdown) -----\n")
print(METHODS)

## 6. Next steps

* **No errors?** The dataset is valid BIDS. Warnings are worth a read, but none of them make it invalid.
* **Sharing the dataset outside the lab** (e.g. on OpenNeuro): use the defaced MRI copies from notebook 03
  in place of the `anat/` images, and keep `DAYSBACK` (notebook 04) private.